# Comparing groups: when is a difference worth acting on?

**Master's in Business Data Science · Module 1 · Session 03, part 2**

Part 1 ended on a box plot showing that genres differ in energy. This notebook asks the
question a curation team would ask next: is that difference real, and is it big enough to
do anything about?

Those are two questions, not one. Statistics answers the first. Only you can answer the
second, and on a table this size the first one turns out to be almost free.

## What we do today, part 2

| | |
|---|---|
| 1 | Look at the two groups before testing anything |
| 2 | A t-test, and what it actually answers |
| 3 | The same test on a difference you cannot see |
| 4 | Effect size, which is the number the p-value will not give you |
| 5 | Why sample size did all of this |
| 6 | More than two groups, and two categorical columns |
| 7 | How to report it |


In [ ]:
# Setup. Same data file as session 02, same loading pattern.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

print(f"pandas {pd.__version__}")

SONGS_URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"
candidates = [Path("data/M1_2026/spotify_songs.csv"), Path("../data/M1_2026/spotify_songs.csv"),
              Path("ds-master/data/M1_2026/spotify_songs.csv"), Path("../ds-master/data/M1_2026/spotify_songs.csv")]

songs = None
for path in candidates:
    if path.exists():
        songs = pd.read_csv(path)
        print(f"Loaded the class snapshot from {path}")
        break
if songs is None:
    songs = pd.read_csv(SONGS_URL)
    print("Loaded the class snapshot from the class URL")

RENAMES = {"track_name": "title", "track_artist": "artist", "track_popularity": "popularity",
           "playlist_genre": "genre", "playlist_subgenre": "subgenre"}
TRACK_COLUMNS = ["track_id", "title", "artist", "energy", "danceability", "loudness", "valence",
                 "tempo", "duration_ms", "acousticness", "instrumentalness", "speechiness",
                 "popularity", "mode"]

print(f"Raw shape: {songs.shape[0]:,} rows x {songs.shape[1]} columns")


### The table from session 02 again

Same cell as part 1.


In [ ]:
# Last week's table, rebuilt in one cell. Nothing here is new.
# One row = one recorded track-playlist-genre association, exactly as in session 02.
membership = songs.rename(columns=RENAMES)[["track_id", "playlist_id", "genre"]].drop_duplicates()
tracks = songs.rename(columns=RENAMES).drop_duplicates(subset=["track_id"])[TRACK_COLUMNS]

joined = membership.merge(tracks, on="track_id", how="left", validate="many_to_one")
assert len(joined) == len(membership)

print(f"Associations (one row each): {len(joined):,}")
print(f"Distinct tracks behind them: {tracks['track_id'].nunique():,}")


## 1. Look first

Two genres, one column. Before any test: how many rows, what are the means, and what do the
distributions look like.


In [ ]:
pair = ["edm", "rock"]
two = joined[joined["genre"].isin(pair)]

display(two.groupby("genre")["energy"].agg(associations="size", mean="mean", sd="std").round(3))


In [ ]:
two.boxplot(column="energy", by="genre", figsize=(7, 4), grid=False)
plt.suptitle("")
plt.title("Energy: edm against rock")
plt.ylabel("energy")
plt.show()


Means of about 0.80 and 0.73, so a gap of roughly 0.07 on a scale that runs from 0 to 1. The
boxes overlap a great deal. Write down now, before running anything: do you think that gap
is real, and do you think it matters?


## 2. A t-test, and what it answers

A two-sample t-test answers one narrow question: if these two groups really had the same
mean, how surprising would a gap this large be?

It does not answer whether the gap is large. It does not answer whether it matters. Keep
those separate in your head for the rest of the notebook.


In [ ]:
edm_energy = joined.loc[joined["genre"] == "edm", "energy"].dropna()
rock_energy = joined.loc[joined["genre"] == "rock", "energy"].dropna()

# equal_var=False is Welch's t-test: it does not assume the two groups have the same spread.
result = stats.ttest_ind(edm_energy, rock_energy, equal_var=False)

print(f"n: {len(edm_energy):,} edm against {len(rock_energy):,} rock")
print(f"means: {edm_energy.mean():.3f} and {rock_energy.mean():.3f}")
print(f"difference: {edm_energy.mean() - rock_energy.mean():.3f}")
print(f"t statistic: {result.statistic:.1f}")
print(f"p value: {result.pvalue:.2e}")


A p value around ten to the power of minus ninety. Read literally: if edm and rock really had
identical mean energy, a gap this size in samples this size would be so rare that the number
has no everyday meaning.

That feels like a strong result. Hold onto that feeling for one more cell.


## 3. The same test, on a difference you cannot see

Now pop against latin. Look at the means first.


In [ ]:
pop_energy = joined.loc[joined["genre"] == "pop", "energy"].dropna()
latin_energy = joined.loc[joined["genre"] == "latin", "energy"].dropna()

print(f"means: {pop_energy.mean():.4f} and {latin_energy.mean():.4f}")
print(f"difference: {pop_energy.mean() - latin_energy.mean():.4f}")


A difference of about one hundredth of a point, on an index bounded between 0 and 1. Nothing
you could hear, nothing anybody would act on, nothing that would survive a different snapshot
of playlists.

Run the same test on it.


In [ ]:
small = stats.ttest_ind(pop_energy, latin_energy, equal_var=False)
print(f"n: {len(pop_energy):,} pop against {len(latin_energy):,} latin")
print(f"difference: {pop_energy.mean() - latin_energy.mean():.4f}")
print(f"p value: {small.pvalue:.4f}")
print("Significant at the 5 percent level?", small.pvalue < 0.05)


Significant. Comfortably so, at conventional thresholds, and a paper reporting "p less than
0.01" for this would be reporting it accurately.

So the word "significant" has now been earned by a gap of 0.068 and by a gap of 0.0095. The
p value cannot tell those two apart, because that is not the question it answers.


> **Judgement call.** This is the moment worth remembering from the whole session. A p value is a statement about
whether a difference exists, given your sample size. It is silent about whether the
difference is large enough to matter. Ask a model to "test whether these genres differ" and
you will get a p value and a verdict. The follow-up question, whether anybody should care,
is not in the output and was not in the request.


## 4. Effect size

The number that separates those two results is the effect size: how big the gap is, measured
in units of how spread out the data already was.

Cohen's d is the common one for two means. It is the difference between the means divided by
the pooled standard deviation, which is four lines of arithmetic.


In [ ]:
def cohens_d(a, b):
    """Difference in means, expressed in pooled standard deviations."""
    pooled_sd = np.sqrt((a.std() ** 2 + b.std() ** 2) / 2)
    return (a.mean() - b.mean()) / pooled_sd


print(f"edm against rock:   d = {cohens_d(edm_energy, rock_energy):.3f}")
print(f"pop against latin:  d = {cohens_d(pop_energy, latin_energy):.3f}")


The usual rough labels are 0.2 for small, 0.5 for medium and 0.8 for large. They are
conventions, not laws, and they are worth knowing mainly so you can see how far below them
most real results sit.

edm against rock lands around 0.4: a real, moderate difference, roughly what you would guess
from the box plot. pop against latin lands around 0.06, which is nothing. Same p-value
verdict, entirely different worlds.

Effect size is what you report to somebody who has to make a decision. The p value is what
you report to somebody who asks whether you could be looking at noise.


In [ ]:
# Every genre pair at once, with both numbers side by side.
rows = []
genres = sorted(joined["genre"].unique())
for i, first in enumerate(genres):
    for second in genres[i + 1:]:
        a = joined.loc[joined["genre"] == first, "energy"].dropna()
        c = joined.loc[joined["genre"] == second, "energy"].dropna()
        rows.append({"pair": f"{first} vs {second}", "difference": a.mean() - c.mean(),
                     "p_value": stats.ttest_ind(a, c, equal_var=False).pvalue,
                     "cohens_d": cohens_d(a, c)})

pairs = pd.DataFrame(rows).sort_values("cohens_d", key=abs, ascending=False)
display(pairs.round({"difference": 3, "cohens_d": 3}).assign(
    p_value=lambda t: t["p_value"].map("{:.1e}".format)))


Fifteen pairs. Nearly all of them are significant, and their effect sizes run from about 1.0
down to near zero. Sorting by effect size rather than by p value puts them in the order a
decision-maker would want.


## 5. Why sample size did all of this

The reason everything is significant is not that music is lawful. It is that each group has
about five thousand rows.

Take the pop against latin comparison, the one with no real difference, and run it on small
samples instead.


In [ ]:
# The same comparison, on 30 rows per group, repeated 200 times.
p_values = []
for seed in range(200):
    a = pop_energy.sample(30, random_state=seed)
    c = latin_energy.sample(30, random_state=seed + 1000)
    p_values.append(stats.ttest_ind(a, c, equal_var=False).pvalue)

p_values = np.array(p_values)
print(f"median p value: {np.median(p_values):.3f}")
print(f"share of runs significant at 5 percent: {(p_values < 0.05).mean():.1%}")


On thirty rows a side, the difference that was "highly significant" a moment ago disappears
almost every time.

The difference in the data never changed. The only thing that changed was how many rows we
looked at. That is the whole mechanism: a p value is a statement about a difference *and*
your sample size, and with enough rows any difference that is not exactly zero will eventually
cross any threshold you pick.


In [ ]:
# Same demo, three sample sizes.
for n in (30, 100, 500):
    ps = np.array([stats.ttest_ind(pop_energy.sample(n, random_state=s),
                                   latin_energy.sample(n, random_state=s + 1000),
                                   equal_var=False).pvalue for s in range(200)])
    print(f"n = {n:4d} per group: median p {np.median(ps):.3f}, "
          f"significant in {(ps < 0.05).mean():5.1%} of runs")


> **Judgement call.** Business data is usually big, so "is it significant" stops being an interesting question
almost immediately. The interesting questions are how large the difference is, whether it is
stable when you cut the data differently, and whether it is large enough to change what
anybody does. None of those are what a significance test returns.


## 6. More than two groups, and two categorical columns

Two more tests, both interpreted the same way as above: the verdict is cheap, the size is the
finding.

### One-way ANOVA: do the six genres differ at all?


In [ ]:
groups = [group["energy"].dropna().values for _, group in joined.groupby("genre")]
anova = stats.f_oneway(*groups)
print(f"F statistic: {anova.statistic:.1f}")
print(f"p value: {anova.pvalue:.2e}")


In [ ]:
# The size that goes with it: how much of the variation in energy is between genres.
grand_mean = joined["energy"].mean()
between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
total = ((joined["energy"] - grand_mean) ** 2).sum()
print(f"eta squared: {between / total:.3f}")


Genre explains about 14 percent of the variation in energy. Which means roughly 86 percent of
it is variation *within* genres: the spread between two edm tracks is usually larger than the
gap between the average edm track and the average rock track.

That single sentence is a better summary of this dataset than any p value in this notebook,
and it is the sort of thing a curation team can act on.

### Chi-square: are genre and musical mode related?

`mode` is 1 for major and 0 for minor. Both columns are categorical, so the test is different
and the reasoning is identical.


In [ ]:
# Step 1 - the contingency table, then the same table as shares.
table = pd.crosstab(joined["genre"], joined["mode"])
table.columns = ["minor", "major"]
display(table)
display(table.div(table.sum(axis=1), axis=0).round(3))


In [ ]:
# Step 2 - the test.
chi2, p_value, dof, expected = stats.chi2_contingency(table)
print(f"chi-square: {chi2:.1f} on {dof} degrees of freedom")
print(f"p value: {p_value:.2e}")

# Step 3 - and its effect size. Cramer's V is chi-square scaled to run from 0 to 1.
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print(f"Cramer's V: {cramers_v:.3f}")


Significant again, and a Cramer's V around 0.12, which is weak. But look at the share table
rather than the single number: rock is about 30 percent minor while edm is about 48 percent.
That gap is worth a sentence, and the overall V is weak partly because the other four genres
sit close together and dilute it.

A single effect-size number for a six-by-two table is a summary, and summaries hide things.
The share table is what you would put in front of somebody.


> **Judgement call.** Three tests, three tiny p values, and the honest write-up of all three is "the differences
are real and mostly small, except for rock, which is genuinely different from the rest". You
would not get that sentence from the tests. You would get it from looking at the tables, which
is what part 1 was for.


## 7. How to report it

A rule worth keeping for the rest of the degree. Never report a p value on its own. Report
three things together:

1. the size of the difference, in the units of the thing you measured;
2. how many observations it rests on;
3. the p value, last, as the smallest of the three claims.

"Mean energy is 0.80 for edm and 0.73 for rock, across 5,899 and 4,866 associations, a gap of
0.068 standard-deviation-scaled to d = 0.40 (p < 0.001)" is a sentence somebody can argue
with. "The difference was significant (p < 0.001)" is not.

## Practice

**Practice 1.** Compare `danceability` between rap and rock. Report the two means, the counts,
Cohen's d and the p value, then write one sentence saying whether a playlist team should care.

**Practice 2.** Pick any genre pair whose energy effect size is below 0.1 in the table from
section 4. Run the small-sample demo on it at n = 100. Explain in two sentences why the p
value moved so much when the data did not.


In [ ]:
# Practice 1. Your turn.
# 1. Pull the two groups.
# 2. Means, counts, d, p.
# 3. One sentence for a playlist team.


In [ ]:
# Practice 2. Your turn.
# 1. Pick a pair with a small d.
# 2. Repeat the test on samples of 100 per group.
# 3. Two sentences on why the p value moved.


## Solutions


In [ ]:
# Solution 1.
rap_dance = joined.loc[joined["genre"] == "rap", "danceability"].dropna()
rock_dance = joined.loc[joined["genre"] == "rock", "danceability"].dropna()
test = stats.ttest_ind(rap_dance, rock_dance, equal_var=False)

print(f"rap:  n = {len(rap_dance):,}, mean {rap_dance.mean():.3f}")
print(f"rock: n = {len(rock_dance):,}, mean {rock_dance.mean():.3f}")
print(f"difference: {rap_dance.mean() - rock_dance.mean():.3f}")
print(f"Cohen's d: {cohens_d(rap_dance, rock_dance):.2f}")
print(f"p value: {test.pvalue:.2e}")


Model answer: rap sits near 0.72 and rock near 0.52, a gap of about 0.20 with an effect size
around 1.0, which is large by any convention and visible in the box plot without a test. Yes,
a playlist team should care: this is one of the few differences in the dataset big enough to
plan around. Note that this is also the case where the p value adds least, because nothing
about the conclusion depends on it.


In [ ]:
# Solution 2, using pop against latin.
for n in (100,):
    ps = np.array([stats.ttest_ind(pop_energy.sample(n, random_state=s),
                                   latin_energy.sample(n, random_state=s + 1000),
                                   equal_var=False).pvalue for s in range(200)])
    print(f"n = {n} per group: median p {np.median(ps):.3f}, "
          f"significant in {(ps < 0.05).mean():.1%} of runs")
print(f"full data:            p {stats.ttest_ind(pop_energy, latin_energy, equal_var=False).pvalue:.4f}")


Model answer: the underlying difference is about 0.01 either way, and at 100 rows per group
the ordinary variation between two random samples is larger than that, so the test cannot
separate the two groups and the p value lands wherever chance puts it. On the full data the
same 0.01 clears the threshold, because with five thousand rows a side the sampling noise
finally becomes smaller than the difference. Nothing about the music changed between those two
runs.

## Takeaways

- A p value answers whether a difference exists, given your sample size. It says nothing about
  whether the difference is big.
- On business-sized tables, almost everything is significant. Effect size becomes the number
  that carries the finding.
- Report the size, the counts, and the p value, in that order.
- Genre explains about 14 percent of the variation in energy in this snapshot. Most of the
  variation is within genres, not between them.
- The tests in this notebook describe one 2020 playlist sample. They are not evidence about
  music, listeners, or anything happening now.
